In [1]:
from tensorflow.keras.layers import BatchNormalization
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import AveragePooling2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers.convolutional import ZeroPadding2D
from keras.layers.core import Activation
from keras.layers.core import Dense
from keras.layers import Flatten
from keras.layers import Input
from keras.models import Model
from keras.layers import add
from keras.regularizers import l2
from keras import backend as K

2021-10-20 21:04:53.345584: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2021-10-20 21:04:53.345638: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [20]:
class ResNet:
    @staticmethod
    def residual_module(data, K, stride, chanDim, red=False,
                       reg=0.0001, bnEps=2e-5, bnMom=0.9):
        '''
        data: input to the residual module
        K: number of filters that will be learned by the final conv layer (the first two
            conv layers will learn K/4 filters)
        stride: controls the stride of the convolution (will help us reduce spatial
            dimensions without using max pooling)
        chanDim: defines the axis which will perform batch normalization
        red: or reduce will control whether we are reducing spatial dimensions (True) or
            not (False) as not all residual modules will reduce dimensions of our spatial volume
        reg: applies regularization strength for all conv layers in the residual modules
        bnEps: controls the epsilon responsible for avoiding 'dividing by zero' erros when
            normalizing inputs
        bnMom: controls the momentum for the moving average
        '''
        
        # initialize as the input (identity) data
        shortcut = data
        
        # the first block of the ResNet module are the 1x1 CONVs
        bn1 = BatchNormalization(axis=chanDim, epsilon=bnEps, momentum=bnMom)(data)
        act1 = Activation("relu")(bn1)
        conv1 = Conv2D(int(K * 0.25), (1, 1), use_bias=False, kernel_regularizer=l2(reg))(act1)
        
        # the second block of the ResNet module are the 3x3 CONVs
        bn2 = BatchNormalization(axis=chanDim, epsilon=bnEps, momentum=bnMom)(conv1)
        act2 = Activation("relu")(bn2)
        conv2 = Conv2D(int(K * 0.25), (3, 3), strides=stride, padding="same", use_bias=False, kernel_regularizer=l2(reg))(act2)
        
        # the third block of the ResNet module is another set of 1x1 CONVs
        bn3 = BatchNormalization(axis=chanDim, epsilon=bnEps, momentum=bnMom)(conv2)
        act3 = Activation("relu")(bn3)
        conv3 = Conv2D(K, (1, 1), use_bias=False, kernel_regularizer=l2(reg))(act3)
        
        if red:
            shortcut = Conv2D(K, (1, 1), strides=stride, use_bias=False, kernel_regularizer=l2(reg))(act1)
            
        x = add([conv3, shortcut])
        
        return x
    
    @staticmethod
    def build(width, height, depth, classes, stages, filters,
             reg=0.0001, bnEps=2e-5, bnMom=0.9):
        
        # initialize the input shape to be 'channels last' and the channels dimensions itself
        inputShape = (height, width, depth)
        chanDim = -1
        
        if K.image_data_format() == "channels_first":
            inputShape = (depth, height, width)
            chanDim = 1
            
        inputs = Input(shape=inputShape)
        x = BatchNormalization(axis=chanDim, epsilon=bnEps, momentum=bnMom)(inputs)
        
        # apply CONV => BN => ACT => POOL to reduce spatial size
        x = Conv2D(filters[0], (5, 5), use_bias=False, padding="same", kernel_regularizer=l2(reg))(x)
        
        x = BatchNormalization(axis=chanDim, epsilon=bnEps, momentum=bnMom)(x)
        
        x = Activation("relu")(x)
        
        x = ZeroPadding2D((1, 1))(x)
        
        x = MaxPooling2D((3, 3), strides=(2, 2))(x)
        
        # loop over the stages
        for i in range(0, len(stages)):
            stride = (1, 1) if i == 0 else (2, 2)
            
            x = ResNet.residual_module(x, filters[i + 1], stride,
                                      chanDim, red=True, bnEps=bnEps, bnMom=bnMom)
            
            # loop over the number of layers in the stage
            for j in range(0, stages[i] - 1):
                x = ResNet.residual_module(x, filters[i + 1], (1, 1), chanDim,
                                          bnEps=bnEps, bnMom=bnMom)
                
        x = BatchNormalization(axis=chanDim, epsilon=bnEps, momentum=bnMom)(x)
        x = Activation("relu")(x)
        x = AveragePooling2D((8, 8))(x)
        
        # softmax classifier
        x = Flatten()(x)
        x = Dense(classes, kernel_regularizer=l2(reg))(x)
        x = Activation("softmax")(x)
        
        model = Model(inputs, x, name="resnet")
        
        return model

In [22]:
model = ResNet.build(width=32, height=32, depth=3, classes=10, stages=(3,4,6,3),
                    filters=(64, 256, 512, 1024, 2048))

ValueError: Negative dimension size caused by subtracting 8 from 2 for '{{node average_pooling2d_1/AvgPool}} = AvgPool[T=DT_FLOAT, data_format="NHWC", ksize=[1, 8, 8, 1], padding="VALID", strides=[1, 8, 8, 1]](Placeholder)' with input shapes: [?,2,2,2048].

In [1]:
import json

In [14]:
with open('./bdd100k_det_20_labels_trainval/labels/det_20/det_train.json') as f:
    data = json.load(f)

In [22]:
print(json.dumps(data[0], indent=2))

{
  "name": "0000f77c-6257be58.jpg",
  "attributes": {
    "weather": "clear",
    "timeofday": "daytime",
    "scene": "city street"
  },
  "timestamp": 10000,
  "labels": [
    {
      "id": "0",
      "attributes": {
        "occluded": false,
        "truncated": false,
        "trafficLightColor": "G"
      },
      "category": "traffic light",
      "box2d": {
        "x1": 1125.902264,
        "y1": 133.184488,
        "x2": 1156.978645,
        "y2": 210.875445
      }
    },
    {
      "id": "1",
      "attributes": {
        "occluded": false,
        "truncated": false,
        "trafficLightColor": "G"
      },
      "category": "traffic light",
      "box2d": {
        "x1": 1156.978645,
        "y1": 136.637417,
        "x2": 1191.50796,
        "y2": 210.875443
      }
    },
    {
      "id": "2",
      "attributes": {
        "occluded": false,
        "truncated": false,
        "trafficLightColor": "NA"
      },
      "category": "traffic sign",
      "box2d": {
    

In [25]:
print(json.dumps(data[0]['labels'], indent=2))

[
  {
    "id": "0",
    "attributes": {
      "occluded": false,
      "truncated": false,
      "trafficLightColor": "G"
    },
    "category": "traffic light",
    "box2d": {
      "x1": 1125.902264,
      "y1": 133.184488,
      "x2": 1156.978645,
      "y2": 210.875445
    }
  },
  {
    "id": "1",
    "attributes": {
      "occluded": false,
      "truncated": false,
      "trafficLightColor": "G"
    },
    "category": "traffic light",
    "box2d": {
      "x1": 1156.978645,
      "y1": 136.637417,
      "x2": 1191.50796,
      "y2": 210.875443
    }
  },
  {
    "id": "2",
    "attributes": {
      "occluded": false,
      "truncated": false,
      "trafficLightColor": "NA"
    },
    "category": "traffic sign",
    "box2d": {
      "x1": 1105.66915985699,
      "y1": 211.122087,
      "x2": 1170.79037,
      "y2": 233.566141
    }
  },
  {
    "id": "3",
    "attributes": {
      "occluded": false,
      "truncated": true,
      "trafficLightColor": "NA"
    },
    "category":

In [32]:
print(json.dumps(data[0]['labels'][4], indent=2))

{
  "id": "4",
  "attributes": {
    "occluded": false,
    "truncated": false,
    "trafficLightColor": "NA"
  },
  "category": "car",
  "box2d": {
    "x1": 49.44476737704903,
    "y1": 254.530367,
    "x2": 357.805838,
    "y2": 487.906215
  }
}


In [36]:
print(json.dumps(data[0]['labels'][4]['category'], indent=2))

"car"


In [34]:
data[0]['labels'][4]['category'] == 'car'

True